In [1]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon')

# 2. Load dataset
df = pd.read_csv("XOM_vs_ORCL_Stock_Report_Data.csv")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print(df.head())
print(df.info())

# 3. NLP sentiment analysis
text_cols = [c for c in df.columns if any(
    w in c for w in ["text", "news", "headline", "title", "description", "content"]
)]

df["text"] = df[text_cols].fillna("").astype(str).agg(" ".join, axis=1) if text_cols else ""
sia = SentimentIntensityAnalyzer()
df["sentiment"] = df["text"].apply(
    lambda x: sia.polarity_scores(x)["compound"]
)

# 4. Convert text into TF-IDF features
if df["text"].str.strip().ne("").any():
    tfidf = TfidfVectorizer(max_features=20, stop_words="english")
    text_features = tfidf.fit_transform(df["text"]).toarray()
    text_df = pd.DataFrame(text_features, index=df.index)
else:
    text_df = pd.DataFrame(index=df.index)

# 5. Prepare numeric features
numeric = df.select_dtypes(include=np.number).replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

X = pd.concat([numeric, text_df], axis=1)
X = X.loc[:, ~X.columns.duplicated()]

# 6. K-Means clustering
X_scaled = StandardScaler().fit_transform(X)
df["cluster"] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_scaled)

# 7. Visualize clusters using PCA
pca = PCA(n_components=2)
points = pca.fit_transform(X_scaled)

sns.scatterplot(x=points[:, 0], y=points[:, 1], hue=df["cluster"], palette="Set1")
plt.title("XOM vs ORCL: K-Means Clusters")
plt.show()

# 8. Sentiment comparison
print("\nAverage sentiment:")
print(df.groupby("cluster")["sentiment"].mean())

ticker_col = next((c for c in df.columns if "ticker" in c or "symbol" in c), None)
if ticker_col:
    print("\nStock comparison:")
    print(df.groupby(ticker_col)["sentiment"].mean())

# 9. Save results
df.to_csv("XOM_vs_ORCL_KMeans_NLP_Results.csv", index=False)
print("\nAnalysis complete! Results saved.")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
/Users/adnanaltimeemy/miniconda3/envs/coding/lib/python3.12/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


      category                 metric  exxonmobil_(xom)      oracle_(orcl)  \
0  Market data                 Ticker               XOM               ORCL   
1  Market data      Latest price used           $164.05            $143.56   
2  Market data  Market capitalisation  About $675–676bn   About $435–438bn   
3  Market data     Shares outstanding      About 4.11bn  About 3.02–3.03bn   
4  Market data            TTM revenue         $361.06bn           $71.78bn   

                                   notes  
0                            NYSE-listed  
1  XOM 8 Oct 2026; ORCL 7 Oct 2026 close  
2    Approximate third-party market data  
3                            Approximate  
4                Third-party market data  
<class 'pandas.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   category          28 non-null     str  
 1   metric            28 non-null     str  
 2

ValueError: n_components=2 must be between 0 and min(n_samples, n_features)=1 with svd_solver='covariance_eigh'